# S4_01 — Text Chunking Strategies

> **Skilljar source**: Section "RAG and Agentic Search", Lesson **L02 — Text chunking strategies**
> **Week_05.md mapping**: §1.2 "청킹 전략" (Week_05.md)
> **Original file**: `001_chunking.ipynb` (downloaded from Skilljar)

## What You Will Learn

1. Why we chunk long documents before feeding them to an embedding model (context-window limits, retrieval granularity, cost).
2. Three concrete chunking strategies and when to use each:
   - **Size-based** — fixed-width windows with overlap
   - **Sentence-based** — N sentences per chunk
   - **Structure-based** — split on Markdown `## ` headers

## Prerequisites

- `report.md` present in the same folder (provided with this download). It is the shared corpus used across **all** S4 notebooks (S4_01 → S4_05).
- No API key needed for this notebook (pure Python string manipulation).

## Connection to Week_05.md

| Week_05.md Section | This Notebook |
|---|---|
| §1.2 "전략 ①: Size-Based" | Cell 1 — `chunk_by_char` |
| §1.2 "전략 ②: Sentence-Based" | Cell 2 — `chunk_by_sentence` |
| §1.2 "전략 ③: Structure-Based" | Cell 3 — `chunk_by_section` |
| §1.2 "전략 ④: Semantic-Based" | *Described in Week_05.md but not implemented in this notebook — covered conceptually only.* |

> [!tip] Reading the code
> All three functions take a plain text string and return a `list[str]`. This uniform signature is deliberate — it lets us swap strategies without changing downstream code in S4_02/S4_03.

## Strategy 1 · Size-Based Chunking (`chunk_by_char`)

Splits text into fixed-character windows with a configurable overlap.

**Parameters**:
- `chunk_size=150` — characters per chunk (tune to your embedding model's token budget)
- `chunk_overlap=20` — characters shared between adjacent chunks

> [!finding] Why overlap?
> Without overlap, a sentence spanning two chunks gets cut. Overlap preserves **local context** at boundaries so an embedding model does not misinterpret a truncated phrase.

**Week_05.md §1.2 — 전략 ①** explains this trade-off: small chunks improve retrieval precision but risk losing cross-boundary meaning; large chunks preserve context but dilute semantic focus.

In [ ]:
# Chunk by a set number of charactesr
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))

        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)

        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )

    return chunks

## Strategy 2 · Sentence-Based Chunking (`chunk_by_sentence`)

Splits on sentence-terminating punctuation (`.`, `!`, `?`) then groups `max_sentences_per_chunk` sentences together.

**Parameters**:
- `max_sentences_per_chunk=5`
- `overlap_sentences=1` — last sentence of chunk N repeats as first of chunk N+1

> [!tip] When to prefer this strategy
> Use when your text is **prose without clear section headers** — articles, transcripts, emails. It guarantees chunk boundaries fall at natural sentence breaks (no mid-sentence cuts).

**Week_05.md §1.2 — 전략 ③** notes the limitation: very long sentences (legal / scientific text) still end up isolated in their own chunk.

In [23]:
# Chunk by sentence
import re


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))

        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))

        start_idx += max_sentences_per_chunk - overlap_sentences

        if start_idx < 0:
            start_idx = 0

    return chunks

## Strategy 3 · Structure-Based Chunking (`chunk_by_section`)

Uses the document's own headings (Markdown `## `) as split points.

> [!tip] Best choice for structured documents
> `report.md` has 13 `## Section N: …` headings → 13 chunks, each a self-contained thematic unit. This is the **highest-fidelity** chunking for retrieval: each chunk's boundary carries real semantic meaning.

**Week_05.md §1.2 — 전략 ②** recommends this for Markdown / HTML / JSON sources where the author already defined logical units.

> [!finding] Generalizing to other formats
> Replace the regex `\n## ` with the heading convention of your corpus (`<h2>` for HTML, custom delimiter for your own CMS).

In [24]:
# Chunk by section
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

## Test Run — Apply Size-Based Chunking to `report.md`

Reads the sample corporate report and prints every size-based chunk (~150 chars each) separated by `----`.

**Expected output**: dozens of short chunks. Notice how sentences and even words get cut mid-way — contrast this with what you would see if you swapped to `chunk_by_section` below.

> [!action] Try it yourself
> 1. Re-run with `chunks = chunk_by_section(text)` — you should get ~13 chunks, each a full section.
> 2. Re-run with `chunks = chunk_by_sentence(text)` — you should get dozens of chunks but none cut mid-sentence.
>
> Your observations here motivate the design choices made in **S4_02** (embeddings) and **S4_03** (vector index).

In [ ]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_char(text)

[print(chunk + "\n----\n") for chunk in chunks]

## Wrap-up

You now have three reusable chunking utilities. In the next notebook (**S4_02**) we will feed these chunks into VoyageAI's embedding API to convert each chunk into a numeric vector.

> [!ref] Skilljar L02 — Text chunking strategies
> Week_05.md §1.2 "청킹 전략" · Figure: `assets/skilljar-s4/L02-*.jpg`